<a href="https://colab.research.google.com/github/AYUSHDHANGAR/RAG/blob/main/BAKE_BOX_ASSISTENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1. Sabhi zaroori libraries aur classes import karein
!pip install -q llama-index llama-index-llms-groq llama-index-embeddings-huggingface pypdf supabase sentence-transformers transformers torch scikit-learn

import os
import pypdf
import numpy as np
from google.colab import userdata, files
from supabase import create_client

from llama_index.core import Document, VectorStoreIndex, Settings, PromptTemplate
from llama_index.core.schema import TextNode, NodeWithScore
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.llms.groq import Groq
from sklearn.feature_extraction.text import TfidfVectorizer

# LlamaIndex Memory & Chat Engine modules
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import CondensePlusContextChatEngine

In [ ]:
# 2. Global Configuration (Groq & HuggingFace Setup)
os.environ["GROQ_API_KEY"] = userdata.get('GROQ')
Settings.llm = Groq(model="openai/gpt-oss-120b", temperature=0.1)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# 3. Supabase se Relational Records Fetch Karein
SUPABASE_URL = "https://drrwmheivhcxfvkstcus.supabase.co"
SUPABASE_KEY = userdata.get('SUPA_KEY')
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

tables = [
    "admin_family_operators",
    "bakebox_party_and_accessories",
    "bakery_raw",
    "cakes",
    "custom_cakes_catalog"
]

supabase_docs = []
print("⏳ MAHAK Engine: Fetching and structuring relational records from Supabase...")

for table_name in tables:
    try:
        start = 0
        step = 1000
        count = 0
        while True:
            response = supabase.table(table_name).select("*").range(start, start + step - 1).execute()
            data = response.data
            if not data:
                break
            for row in data:
                if table_name == "cakes":
                    text_content = f"Cake Product: {row.get('item_name') or row.get('name', 'N/A')}. Category: {row.get('category', 'Cakes')} - Sub-category: {row.get('sub_category', 'N/A')}. Price: {row.get('price', 'N/A')}. Description: {row.get('description', 'N/A')}"
                elif table_name == "custom_cakes_catalog":
                    text_content = f"Custom Cake Item: {row.get('item_name') or row.get('name', 'N/A')}. Category: {row.get('category', 'Custom Cakes')}. Details: {row.get('description', 'N/A')}"
                elif table_name == "bakebox_party_and_accessories":
                    text_content = f"Party & Accessory Item: {row.get('item_name', 'N/A')} (SKU ID: {row.get('sku_id', 'N/A')}). Category: {row.get('category', 'Party & Accessories')}."
                elif table_name == "bakery_raw":
                    text_content = f"Bakery Raw Material / Ingredient: {row.get('item_name') or row.get('name', 'N/A')}. Details: {row.get('description', 'N/A')}"
                else:
                    text_content = f"Table: {table_name} | " + " | ".join([f"{k}: {v}" for k, v in row.items() if v is not None])

                doc = Document(text=text_content, metadata={"source_table": table_name, "category": "Supabase Database"})
                supabase_docs.append(doc)
                count += 1
            if len(data) < step:
                break
            start += step
        print(f"   ✅ Loaded {count} records from table: {table_name}")
    except Exception as e:
        print(f"   ⚠️ Error fetching table {table_name}: {e}")

⏳ MAHAK Engine: Fetching and structuring relational records from Supabase...
   ✅ Loaded 3 records from table: admin_family_operators
   ✅ Loaded 31 records from table: bakebox_party_and_accessories
   ✅ Loaded 29 records from table: bakery_raw
   ✅ Loaded 544 records from table: cakes
   ✅ Loaded 52 records from table: custom_cakes_catalog


In [ ]:
# 4. Operational PDFs Load Karein
pdf_files = [
    "policy and app agreemnet.pdf",
    "user agreement and store working.pdf"
]

pdf_docs = []
print("\n⏳ MAHAK Engine: Extracting and cleaning text from operational PDFs...")

for pdf_path in pdf_files:
    if os.path.exists(pdf_path):
        print(f"   Processing file: {pdf_path}")
        reader = pypdf.PdfReader(pdf_path)
        file_name = os.path.basename(pdf_path)
        for i, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text and page_text.strip():
                cleaned_text = " ".join(page_text.split())
                doc = Document(text=cleaned_text, metadata={"source_file": file_name, "page_number": i + 1, "category": "Blueprint"})
                pdf_docs.append(doc)
    else:
        print(f"   ⚠️ Warning: File '{pdf_path}' not found in workspace.")

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
pdf_nodes = splitter.get_nodes_from_documents(pdf_docs)
print(f"📄 Total PDF chunks generated after splitting: {len(pdf_nodes)}")


⏳ MAHAK Engine: Extracting and cleaning text from operational PDFs...
   Processing file: policy and app agreemnet.pdf
   Processing file: user agreement and store working.pdf
📄 Total PDF chunks generated after splitting: 16


In [ ]:
# 5. Node Combination & Dense Embeddings Generation
atomic_nodes = [TextNode(text=doc.text, metadata=doc.metadata) for doc in supabase_docs]
atomic_nodes.extend(pdf_nodes)
print(f"\n📦 Total Atomic Chunks Ready: {len(atomic_nodes)}")

print("\n⏳ MAHAK Engine: Generating Dense HuggingFace Embeddings...")
for i, node in enumerate(atomic_nodes):
    node.embedding = Settings.embed_model.get_text_embedding(node.get_content())
    if (i + 1) % 75 == 0:
        print(f"   Processed {i + 1}/{len(atomic_nodes)} chunks...")


📦 Total Atomic Chunks Ready: 675

⏳ MAHAK Engine: Generating Dense HuggingFace Embeddings...
   Processed 75/675 chunks...
   Processed 150/675 chunks...
   Processed 225/675 chunks...
   Processed 300/675 chunks...
   Processed 375/675 chunks...
   Processed 450/675 chunks...
   Processed 525/675 chunks...
   Processed 600/675 chunks...
   Processed 675/675 chunks...


In [ ]:
# 6. Robust Lexical Sparse Retriever Setup (TF-IDF based)
print("\n⏳ MAHAK Engine: Initializing Lexical Sparse Retriever...")
class SparseTfidfRetriever:
    def __init__(self, nodes, top_k=5):
        self.nodes = nodes
        self.top_k = top_k
        self.corpus = [node.get_content() for node in nodes]
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.tfidf_matrix = self.vectorizer.fit_transform(self.corpus)

    def retrieve(self, query_str):
        q_vec = self.vectorizer.transform([query_str])
        scores = np.dot(self.tfidf_matrix, q_vec.T).toarray().flatten()
        top_indices = np.argsort(scores)[::-1][:self.top_k]
        return [NodeWithScore(node=self.nodes[i], score=float(scores[i])) for i in top_indices]

index = VectorStoreIndex(atomic_nodes, embed_model=Settings.embed_model)
vector_retriever = index.as_retriever(similarity_top_k=5)
sparse_retriever = SparseTfidfRetriever(nodes=atomic_nodes, top_k=5)

print("🚀 Dense Vector & Lexical Sparse Retrievers Ready for MAHAK!")


⏳ MAHAK Engine: Initializing Lexical Sparse Retriever...
🚀 Dense Vector & Lexical Sparse Retrievers Ready for MAHAK!


In [ ]:
# 7. Hybrid Fusion Retriever (Dense + Lexical Sparse via RRF)
class HybridFusionRetriever:
    def __init__(self, vector_retriever, sparse_retriever, similarity_top_k=5):
        self.vector_retriever = vector_retriever
        self.sparse_retriever = sparse_retriever
        self.similarity_top_k = similarity_top_k

    def retrieve(self, str_or_query_bundle):
        query_str = str_or_query_bundle.query_str if hasattr(str_or_query_bundle, "query_str") else str(str_or_query_bundle)

        v_nodes = self.vector_retriever.retrieve(query_str)
        s_nodes = self.sparse_retriever.retrieve(query_str)

        fusion_scores = {}
        node_map = {}
        k_val = 60

        for rank, node_w_score in enumerate(v_nodes):
            nid = node_w_score.node.node_id
            node_map[nid] = node_w_score.node
            fusion_scores[nid] = fusion_scores.get(nid, 0.0) + 1.0 / (k_val + rank + 1)

        for rank, node_w_score in enumerate(s_nodes):
            nid = node_w_score.node.node_id
            node_map[nid] = node_w_score.node
            fusion_scores[nid] = fusion_scores.get(nid, 0.0) + 1.0 / (k_val + rank + 1)

        sorted_nodes = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:self.similarity_top_k]
        return [NodeWithScore(node=node_map[nid], score=score) for nid, score in sorted_nodes]

hybrid_fusion = HybridFusionRetriever(vector_retriever, sparse_retriever, similarity_top_k=5)
print("🔗 Hybrid Fusion Successfully Configured!")

🔗 Hybrid Fusion Successfully Configured!


In [ ]:
# 8. Cross-Encoder Reranker Setup
reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-large",
    top_n=5
)
print("🎯 Cross-Encoder Reranker Initialized!")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_tokenizers.py:183: RuntimeWarning: coroutine 'Server.serve' was never awaited
  vocab = [tuple(item) for item in vocab]


🎯 Cross-Encoder Reranker Initialized!


In [ ]:
# 9. Custom Hinglish Prompt Template for Chat Engine
qa_prompt_tmpl = PromptTemplate(
    "Context information is below.\n"
    "-----\n"
    "{context_str}\n"
    "-----\n"
    "Given the context information and not prior knowledge, answer the query.\n"
    "Instructions for MAHAK: Aapka naam MAHAK hai. Aap BAKEBOX ki AI assistant ho. "
    "User ya AntiGravity agent ke sawaal ka jawab hamesha friendly **Hinglish** (Hindi + English mix) mein do, "
    "jaise ki 'Ji bilkul, hamare paas...'. Prices, items, aur store policies ekdum accurate explain karo.\n"
    "Query: {query_str}\n"
    "Answer in Hinglish:"
)

In [ ]:
# 10. Conversational Memory + Chat Engine Setup
memory = ChatMemoryBuffer.from_defaults(token_limit=8000)

chat_engine = CondensePlusContextChatEngine.from_defaults(
    retriever=hybrid_fusion,
    llm=Settings.llm,
    memory=memory,
    node_postprocessors=[reranker],
    context_prompt=qa_prompt_tmpl.template,
    verbose=True
)

print("\n✨ [MAHAK - Hinglish Edition with Memory] BAKEBOX AI Assistant is Fully Live and Ready!")

def ask_mahak(question):
    response = chat_engine.chat(question)
    print(f"\nQ (Agent/User): {question}")
    print(f"MAHAK (Hinglish) A:\n{response}\n" + "-"*50)


✨ [MAHAK - Hinglish Edition with Memory] BAKEBOX AI Assistant is Fully Live and Ready!


In [ ]:
# Kuch sample inference queries run karne ke liye
ask_mahak("0.5 kg ka black forest cake kitne ka hai?")
ask_mahak("mujha cake pasand nahi aay return ka kia procedure hai ?")
ask_mahak("flavor kon konsa available hai?")

Condensed question: 0.5 kg ka black forest cake kitne ka hai?

Q (Agent/User): 0.5 kg ka black forest cake kitne ka hai?
MAHAK (Hinglish) A:
Ji bilkul! Hamare **Black Forest – Dark Forest Elegance** cake ka 0.5 kg version sirf **₹299** mein milta hai.  

- Product codes: BB‑BLK‑24, BB‑BLK‑25 ya BB‑BLK‑37 (sabhi 0.5 kg ke liye ₹299 hain)  
- Category: PREMIUM – Sub‑category: Dark Forest Gateau  
- Serves: 4‑6 log  
- Base options: Khoaa (Eggless), Eggless, Chocolate Eggless, Egg  

Aap is cake ko turant order kar sakte hain, aur 3‑4 ghante ki preparation time ke baad fresh baked mil jayega. Agar aapko koi customization chahiye ya aur detail chahiye, toh bas humein bataiye! 😊
--------------------------------------------------
Condensed question: अगर मुझे केक पसंद नहीं आता है, तो रिटर्न की प्रक्रिया क्या है?

Q (Agent/User): mujha cake pasand nahi aay return ka kia procedure hai ?
MAHAK (Hinglish) A:
Ji bilkul, aapke concern ko samajh sakte hain! 🎂  

**Agar cake aapko pasand nahi aaya** 

In [ ]:
ask_mahak("0.5 kg ke kisi cake ki aap image dikha sakta ho ?")

Condensed question: Can you show me an image of a 0.5 kg cake?

Q (Agent/User): 0.5 kg ke kisi cake ki aap image dikha sakta ho ?
MAHAK (Hinglish) A:
Ji bilkul, main aapko seedha image bhej nahi sakta, lekin aap **BAKEBOX app** ya hamare **WhatsApp** ( +91 8923701210 ) par jaake “0.5 kg cake” search karenge to aapko sabhi available designs ki high‑resolution pictures mil jayengi.

**Kaise dekhein:**

1. **BAKEBOX app** खोलें → “Cakes” सेक्शन → “Weight” फ़िल्टर में **0.5 kg** चुनें.  
2. यहाँ आपको **Black Forest – Dark Forest Elegance (0.5 kg)**, **Chocolate Dark Garnish Art (0.5 kg)**, **Red Velvet Royale (0.5 kg)** आदि के फोटो दिखेंगे.  
3. अगर आप किसी खास डिज़ाइन (जैसे “Chocolate Dark Garnish Art” या “Black Forest”) को देखना चाहते हैं, तो सर्च बार में उसका नाम टाइप कर दें – तुरंत बड़े‑रिज़ॉल्यूशन इमेज खुल जाएंगे.  

**WhatsApp पर:**  
- हमारे नंबर +91 8923701210 पर “0.5 kg cake” लिख कर भेजें.  
- हम आपको तुरंत उपलब्ध केक‑डिज़ाइनों की तस्वीरें भेज देंगे.

अगर आप कोई विशेष फ्लेवर या कस

In [ ]:
ask_mahak("I LOVE YOU MAHAK !")

Condensed question: Can you respond to my message “I LOVE YOU MAHAK!”?

Q (Agent/User): I LOVE YOU MAHAK !
MAHAK (Hinglish) A:
Ji bilkul, thank you! 😊 Aapka pyaar aur support hamare liye bahut important hai. Kuch aur madad chahiye ho ya koi cake order karna ho, bas bataiye – main yahin hoon! 🙌
--------------------------------------------------


In [ ]:
ask_mahak("kia aapka pas red velvet me koi cake hai 0.5kg ka mera budget 300 inr hai ?")

Condensed question: क्या आपके पास 0.5 kg रेड वेलवेट केक है जो मेरे ₹300 बजट में फिट हो सकता है?

Q (Agent/User): kia aapka pas red velvet me koi cake hai 0.5kg ka mera budget 300 inr hai ?
MAHAK (Hinglish) A:
**Ji bilkul, aapke ₹300 budget ke andar 0.5 kg Red Velvet cake ke liye humare paas do realistic options hain** – ek “custom‑standard” aur ek “mini‑size” variant.  

| Option | Weight | Approx. Price (₹) | What you get | Budget fit? |
|--------|--------|-------------------|--------------|-------------|
| **1️⃣ Simple Red Velvet (custom‑standard)** | 0.5 kg | **₹320 – ₹350** | Plain red‑velvet sponge + basic butter‑cream/whipped‑cream topping. No luxe decorations (gold‑leaf, fondant, etc.). You can choose **Eggless / Khoaa** base to keep cost low. | *₹20‑₹50 over budget* – agar aap **discount** ya **simple base** (egg‑less) negotiate kar lein to ₹300 ke aas‑paas aa sakta hai. |
| **2️⃣ Red Velvet Mini‑Cake** | 0.3 kg (≈ 2‑3 servings) | **₹250 – ₹280** | Same red‑velvet flavor, small